In [1]:
%matplotlib inline

# import sys, os
# sys.path.append(os.path.abspath(".."))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import spearmanr, wilcoxon
from scipy.cluster.hierarchy import linkage, fcluster, dendrogram
from scipy.spatial.distance import squareform
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
import warnings
warnings.filterwarnings('ignore', category=UserWarning)

from behav_utils.data.structures import ExperimentData, AnimalData
from behav_utils.data.synthetic import generate_synthetic_animal
from behav_utils.data.loading import load_experiment, load_from_directory
from behav_utils.analysis.session_features import (
    build_feature_matrix,
    build_feature_matrix_multi,
    get_feature_columns,
    METADATA_COLUMNS,
    summarise_features,
)
from behav_utils.analysis.summary_stats import FEATURE_MATRIX_STATS

print("Imports OK")
print(f"FEATURE_MATRIX_STATS ({len(FEATURE_MATRIX_STATS)}):")
for s in FEATURE_MATRIX_STATS:
    print(f"  - {s}")

Imports OK
FEATURE_MATRIX_STATS (21):
  - accuracy
  - psychometric
  - psychometric_gof
  - recency
  - stimulus_recency
  - recency_divergence
  - win_stay
  - lose_shift
  - stimulus_sensitivity
  - side_bias
  - choice_autocorr
  - choice_entropy
  - perseveration
  - hard_easy_ratio
  - hard_accuracy
  - easy_accuracy
  - history_interaction_r2
  - sd_profile
  - logistic_history
  - update_matrix
  - conditional_psychometric


# Load Data

In [2]:
path_config = '/Users/Serkan/Desktop/pro/PhD/main/repos/sound_categorisation/config.yaml'
experiment = load_experiment(path_config)
STAGE = 'Full_Task_Cont'
SHIFT_SESSION = None

Loaded 12 animals, 433 total sessions


In [6]:
def get_animal_idx(experiment, animal_id):
	for idx, aid in enumerate(experiment.animal_ids):
		if aid == animal_id:
			return idx
	return None

In [3]:
all_animals = experiment.get_animals(min_sessions=15)

In [9]:
pooled_df = build_feature_matrix_multi(all_animals, stage=STAGE)
print(f"Feature matrix: {pooled_df.shape[0]} sessions × {pooled_df.shape[1]} columns")
print(f"Animals: {pooled_df['animal_id'].nunique()}")
print(f"Sessions per animal: {pooled_df.groupby('animal_id').size().values}")

Feature matrix: 393 sessions × 151 columns
Animals: 10
Sessions per animal: [49 56 41 41 41 41 32 31 31 30]


In [10]:
pooled_df

,animal_id,session_id,session_idx,date,stage,distribution,n_trials_total,n_trials_valid,n_trials_abort,abort_rate,...,rt_median_hard,rt_median_easy,rt_correct_vs_error,delta_pse,delta_slope,delta_accuracy,delta_side_bias,delta_recency,delta_choice_entropy,delta_rt_median
0,SS01,SOUND_CAT_SS01_2026_1_12,1,2026-01-12,Full_Task_Cont,Uniform,594,556,38,0.063973,...,0.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,SS01,SOUND_CAT_SS01_2026_1_13,2,2026-01-13,Full_Task_Cont,Uniform,399,388,11,0.027569,...,0.0,0.0,0.0,NaN,NaN,0.022806,0.081658,0.033521,0.018436,0.0
2,SS01,SOUND_CAT_SS01_2026_1_14,3,2026-01-14,Full_Task_Cont,Uniform,454,430,24,0.052863,...,0.0,0.0,0.0,0.517979,0.041342,0.051558,0.203141,0.032962,0.131820,0.0
3,SS01,SOUND_CAT_SS01_2026_1_15,4,2026-01-15,Full_Task_Cont,Uniform,476,462,14,0.029412,...,0.0,0.0,0.0,0.192927,0.709880,0.075647,0.021655,0.018024,0.109264,0.0
4,SS01,SOUND_CAT_SS01_2026_1_16,5,2026-01-16,Full_Task_Cont,Uniform,395,371,24,0.060759,...,0.0,0.0,0.0,0.047071,0.150833,0.067549,0.152659,0.057650,0.068591,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
388,SS13,SOUND_CAT_SS13_2026_3_18,27,2026-03-18,Full_Task_Cont,Hard-B,352,313,39,0.110795,...,33.0,0.0,-57.5,0.112094,0.191871,0.059956,0.036541,0.282580,0.036952,0.0
389,SS13,SOUND_CAT_SS13_2026_3_20,28,2026-03-20,Full_Task_Cont,Hard-B,251,234,17,0.067729,...,0.0,0.0,-99.0,0.067636,0.010600,0.071667,0.067898,0.019462,0.136236,0.0
390,SS13,SOUND_CAT_SS13_2026_3_23,29,2026-03-23,Full_Task_Cont,Uniform,297,293,4,0.013468,...,0.0,0.0,0.0,0.128835,0.205031,0.043916,0.035982,0.128076,0.047348,0.0
391,SS13,SOUND_CAT_SS13_2026_3_24,30,2026-03-24,Full_Task_Cont,Hard-B,350,312,38,0.108571,...,0.0,0.0,-66.0,0.117999,0.237742,0.052463,0.017437,0.154904,0.108254,0.0


In [19]:
feature_cols = get_feature_columns(pooled_df)
print(f"Numeric feature columns: {len(feature_cols)}")
nan_counts = pooled_df[feature_cols].isna().sum().sort_values(ascending=False)
nan_pct = (nan_counts / len(pooled_df) * 100).round(1)
nan_df = pd.DataFrame({'feature': nan_counts.index, 'n_nan': nan_counts.values,
                        'pct_nan': nan_pct.values})
print("\nNaN prevalence:")
print(nan_df[nan_df['n_nan'] > 5].to_string(index=False))
if nan_df['n_nan'].sum() == 0:
    print("  (all features fully populated)")
    
NAN_THRESHOLD = 0.20
high_nan = nan_df[nan_df['pct_nan'] > NAN_THRESHOLD * 100]['feature'].tolist()
if high_nan:
    print(f"\nExcluding (>{NAN_THRESHOLD*100:.0f}% NaN): {high_nan}")
    
clean_features = [f for f in feature_cols if f not in high_nan]


Numeric feature columns: 134

NaN prevalence:
     feature  n_nan  pct_nan
sd_curvature     62     15.8
       slope     37      9.4
         pse     37      9.4
    sd_range     33      8.4
    sd_slope     33      8.4


In [ ]:
# Also remove near-zero-variance
variances = pooled_df[clean_features].var()
near_zero = variances[variances < 1e-10].index.tolist()
if near_zero:
    print(f"Removing near-zero-variance: {near_zero}")
    clean_features = [f for f in clean_features if f not in near_zero]

print(f"\nRetained features: {len(clean_features)}")